# 🦿 Jev API + Gymnasium BipedalWalker Controller — v2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/simple-jev/blob/main/notebooks/Jev_API_BipedalWalker_Controller_Colab.ipynb)

This version fixes the **jumping / forward-crash bug**.

### What was wrong?
The old notebook asked Jev for a four-motor action every 10 frames and then **held those same motor values for all 10 physics frames**. BipedalWalker needs fresh hip/knee commands every frame (50 Hz), so a normal gait correction became a long impulse: **jump → pitch forward → hull crash**.

### v2 architecture
**Gym observation → 50 Hz reference gait → Jev supervisory mode → motors**

Jev chooses one supervisory mode:
- `nominal`
- `cautious`
- `stabilize`
- `smooth`

The four motor values are recomputed on **every simulator frame**. `DECISION_EVERY` only controls how often Jev changes strategy.

Create/get the API key at **https://typesafe.ai** and add it to Colab Secrets as `TYPESAFE_API_KEY`.


In [ ]:
#@title 1. Install dependencies
!apt-get update -qq
!apt-get install -y -qq swig > /dev/null
!pip -q install "gymnasium[box2d]" imageio imageio-ffmpeg typesafe-sdk


In [ ]:
#@title 2. Imports, API key and configuration
import os, time, json, getpass
from collections import deque

import numpy as np
import gymnasium as gym
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from IPython.display import Video, display

from typesafe_sdk import Choice, TypeSafeClient

def load_typesafe_key():
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("TYPESAFE_API_KEY")
    except Exception:
        pass
    if not key:
        key = os.environ.get("TYPESAFE_API_KEY")
    if not key:
        key = getpass.getpass(
            "Enter TYPESAFE_API_KEY from https://typesafe.ai: "
        ).strip()
    if not key:
        raise RuntimeError("No TYPESAFE_API_KEY supplied.")
    return key

os.environ["TYPESAFE_API_KEY"] = load_typesafe_key()
os.environ.setdefault("TYPESAFE_DEFAULT_MODEL", "jev-latest")
client = TypeSafeClient()

SEED = 0
MAX_STEPS = 1600

# IMPORTANT: Jev supervisory interval only.
# Low-level motor commands are recomputed EVERY Gym frame.
DECISION_EVERY = 10

MIN_CONFIDENCE = 0.15
VIDEO_EVERY = 4
VIDEO_PATH = "/content/jev_bipedalwalker_v2.mp4"
LOG_PATH = "/content/jev_bipedalwalker_v2_decisions.json"

print("✓ TypeSafe key loaded securely")
print("✓ Jev client ready")
print("model:", os.environ["TYPESAFE_DEFAULT_MODEL"])
print("Jev mode decision every", DECISION_EVERY, "frames")
print("motor update: every frame (50 Hz)")


In [ ]:
#@title 3. Test the Jev API key
response = client.system_one(
    state={
        "task": "API connectivity test",
        "walker": {"hull_angle": 0.0, "horizontal_speed": 0.1},
    },
    questions={
        "motor_plan": Choice(
            instructions="Choose the better plan for a stable walking robot.",
            criteria={
                "keep_nominal": "Continue the current stable gait.",
                "stabilize": "Prioritize balance recovery before speed.",
            },
        )
    },
)

answer = response.answers["motor_plan"]
print("✓ Jev API works")
print("choice:", answer.choice)
print("confidence:", round(float(answer.confidence), 3))
print("probabilities:", answer.probabilities)
print("served model:", response.model)


In [ ]:
#@title 4. 50 Hz gait controller + Jev supervisory modes
class ReferenceWalkerController:
    # Same finite-state gait structure used by Gymnasium's BipedalWalker heuristic.
    STAY_ON_ONE_LEG, PUT_OTHER_DOWN, PUSH_OFF = 1, 2, 3
    SPEED = 0.29
    SUPPORT_KNEE_ANGLE = 0.1

    def __init__(self):
        self.state = self.STAY_ON_ONE_LEG
        self.moving_leg = 0
        self.supporting_leg = 1
        self.supporting_knee_angle = self.SUPPORT_KNEE_ANGLE
        self.a = np.zeros(4, dtype=np.float32)

    def step(self, s):
        moving_s_base = 4 + 5 * self.moving_leg
        supporting_s_base = 4 + 5 * self.supporting_leg

        hip_targ = [None, None]
        knee_targ = [None, None]
        hip_todo = [0.0, 0.0]
        knee_todo = [0.0, 0.0]

        if self.state == self.STAY_ON_ONE_LEG:
            hip_targ[self.moving_leg] = 1.1
            knee_targ[self.moving_leg] = -0.6
            self.supporting_knee_angle += 0.03

            if s[2] > self.SPEED:
                self.supporting_knee_angle += 0.03

            self.supporting_knee_angle = min(
                self.supporting_knee_angle,
                self.SUPPORT_KNEE_ANGLE
            )
            knee_targ[self.supporting_leg] = self.supporting_knee_angle

            if s[supporting_s_base] < 0.10:
                self.state = self.PUT_OTHER_DOWN

        if self.state == self.PUT_OTHER_DOWN:
            hip_targ[self.moving_leg] = 0.1
            knee_targ[self.moving_leg] = self.SUPPORT_KNEE_ANGLE
            knee_targ[self.supporting_leg] = self.supporting_knee_angle

            if s[moving_s_base + 4]:
                self.state = self.PUSH_OFF
                self.supporting_knee_angle = min(
                    s[moving_s_base + 2],
                    self.SUPPORT_KNEE_ANGLE
                )

        if self.state == self.PUSH_OFF:
            knee_targ[self.moving_leg] = self.supporting_knee_angle
            knee_targ[self.supporting_leg] = 1.0

            if (
                s[supporting_s_base + 2] > 0.88
                or s[2] > 1.2 * self.SPEED
            ):
                self.state = self.STAY_ON_ONE_LEG
                self.moving_leg = 1 - self.moving_leg
                self.supporting_leg = 1 - self.moving_leg

        if hip_targ[0]:
            hip_todo[0] = 0.9 * (hip_targ[0] - s[4]) - 0.25 * s[5]
        if hip_targ[1]:
            hip_todo[1] = 0.9 * (hip_targ[1] - s[9]) - 0.25 * s[10]
        if knee_targ[0]:
            knee_todo[0] = 4.0 * (knee_targ[0] - s[6]) - 0.25 * s[7]
        if knee_targ[1]:
            knee_todo[1] = 4.0 * (knee_targ[1] - s[11]) - 0.25 * s[12]

        # Hull stabilization and vertical damping from Gymnasium heuristic.
        hip_todo[0] -= 0.9 * (0.0 - s[0]) - 1.5 * s[1]
        hip_todo[1] -= 0.9 * (0.0 - s[0]) - 1.5 * s[1]
        knee_todo[0] -= 15.0 * s[3]
        knee_todo[1] -= 15.0 * s[3]

        self.a[:] = [
            hip_todo[0], knee_todo[0],
            hip_todo[1], knee_todo[1]
        ]
        return np.clip(0.5 * self.a, -1.0, 1.0).astype(np.float32)


MODE_CRITERIA = {
    "nominal": (
        "Use the normal alternating gait. Prefer this when the hull is "
        "reasonably upright and forward progress is stable."
    ),
    "cautious": (
        "Reduce motor intensity while preserving gait timing. Use when "
        "motion is too aggressive or vertical bouncing is increasing."
    ),
    "stabilize": (
        "Prioritize hull stabilization and soften the knees. Use when "
        "hull tilt/angular velocity indicates a developing fall."
    ),
    "smooth": (
        "Blend each fresh gait command with the previous frame. Use when "
        "motion is oscillatory or recent control changes are too abrupt."
    ),
}


def apply_jev_mode(reference_action, obs, previous_action, mode):
    ref = np.asarray(reference_action, dtype=np.float32).copy()
    prev = np.asarray(previous_action, dtype=np.float32).copy()
    effective_mode = mode

    # Frame-level guard: a supervisory decision can be several frames old.
    if abs(float(obs[0])) > 0.32 or abs(float(obs[1])) > 0.30:
        effective_mode = "stabilize"

    if effective_mode == "nominal":
        a = ref

    elif effective_mode == "cautious":
        a = 0.80 * ref

    elif effective_mode == "smooth":
        # Fresh reference action is still computed every frame.
        a = 0.70 * ref + 0.30 * prev

    elif effective_mode == "stabilize":
        a = ref.copy()

        # Reinforce the same hull correction used by the gait controller.
        balance = np.clip(
            0.9 * float(obs[0]) + 1.5 * float(obs[1]),
            -0.45, 0.45
        )
        a[0] += 0.35 * balance
        a[2] += 0.35 * balance

        # Avoid violent knee extension while recovering.
        a[1] *= 0.80
        a[3] *= 0.80

    else:
        effective_mode = "nominal"
        a = ref

    return (
        np.clip(a, -1.0, 1.0).astype(np.float32),
        effective_mode
    )

print("✓ 50 Hz gait controller ready")


In [ ]:
#@title 5. Convert Gym state to Jev supervisory state
def walker_state_for_jev(
    obs,
    step,
    total_reward,
    recent_rewards,
    previous_mode,
):
    r = list(recent_rewards)

    return {
        "task": {
            "environment": "Gymnasium BipedalWalker-v3",
            "goal": (
                "Walk right while staying upright. Avoid hull-ground contact "
                "and bouncing. Select a supervisory gait mode; the low-level "
                "motor controller updates every frame."
            ),
            "decision_horizon_frames": DECISION_EVERY,
        },
        "progress": {
            "step": int(step),
            "total_reward": round(float(total_reward), 3),
            "recent_reward_sum": round(float(sum(r)), 3),
            "recent_reward_mean": round(
                float(np.mean(r)) if r else 0.0, 4
            ),
            "previous_mode": previous_mode,
        },
        "hull": {
            "angle_rad": round(float(obs[0]), 4),
            "angular_velocity_scaled": round(float(obs[1]), 4),
            "horizontal_speed_scaled": round(float(obs[2]), 4),
            "vertical_speed_scaled": round(float(obs[3]), 4),
        },
        "contacts": {
            "leg0": bool(obs[8] > 0.5),
            "leg1": bool(obs[13] > 0.5),
        },
        "legs": {
            "leg0_hip": round(float(obs[4]), 3),
            "leg0_knee": round(float(obs[6]), 3),
            "leg1_hip": round(float(obs[9]), 3),
            "leg1_knee": round(float(obs[11]), 3),
        },
        "terrain_lidar": [
            round(float(x), 3) for x in obs[14:24]
        ],
    }


In [ ]:
#@title 6. Run one Jev-supervised episode — fixed control loop
env = gym.make("BipedalWalker-v3", render_mode="rgb_array")
obs, info = env.reset(seed=SEED)

reference = ReferenceWalkerController()
previous_action = np.zeros(4, dtype=np.float32)
active_mode = "nominal"

recent_rewards = deque(maxlen=30)
total_reward = 0.0

decision_log = []
reward_history = []
speed_history = []
angle_history = []
mode_history = []

fallbacks = 0
api_failures = 0

# H.264 at native 600x400:
# no macro-block resize warning and better browser compatibility.
writer = imageio.get_writer(
    VIDEO_PATH,
    format="FFMPEG",
    mode="I",
    fps=max(1, 50 // VIDEO_EVERY),
    codec="libx264",
    pixelformat="yuv420p",
    macro_block_size=1,
)

try:
    for step in range(MAX_STEPS):

        # ---------------------------------------------------------
        # 1) Jev changes only the supervisory MODE every N frames.
        # ---------------------------------------------------------
        if step % DECISION_EVERY == 0:
            state = walker_state_for_jev(
                obs,
                step,
                total_reward,
                recent_rewards,
                active_mode,
            )

            t0 = time.perf_counter()

            try:
                response = client.system_one(
                    state=state,
                    questions={
                        "control_mode": Choice(
                            instructions=(
                                "Choose one supervisory gait mode for the next "
                                "short interval. Goal: move right, keep the hull "
                                "upright, avoid hull-ground contact and bouncing. "
                                "The low-level four-motor controller runs every "
                                "physics frame, so do not choose frozen motor values."
                            ),
                            criteria=MODE_CRITERIA,
                        )
                    },
                )

                latency_ms = (
                    time.perf_counter() - t0
                ) * 1000.0

                answer = response.answers["control_mode"]
                chosen_mode = answer.choice
                confidence = float(answer.confidence)

                if chosen_mode not in MODE_CRITERIA:
                    raise RuntimeError(
                        f"Unknown Jev mode: {chosen_mode}"
                    )

                used_fallback = False

                if confidence < MIN_CONFIDENCE:
                    chosen_mode = "nominal"
                    used_fallback = True
                    fallbacks += 1

                active_mode = chosen_mode

                decision_log.append({
                    "step": step,
                    "mode": active_mode,
                    "confidence": confidence,
                    "probabilities": dict(answer.probabilities),
                    "latency_ms": latency_ms,
                    "model": response.model,
                    "fallback": used_fallback,
                    "horizontal_speed": float(obs[2]),
                    "vertical_speed": float(obs[3]),
                    "hull_angle": float(obs[0]),
                    "hull_angular_velocity": float(obs[1]),
                    "total_reward": float(total_reward),
                })

                print(
                    f"decision {len(decision_log):03d} | "
                    f"step {step:04d} | "
                    f"{active_mode:>9s} | "
                    f"conf {confidence:.3f} | "
                    f"{latency_ms:.0f} ms | "
                    f"vx {obs[2]:+.3f} | "
                    f"angle {obs[0]:+.3f}"
                    + (" | FALLBACK" if used_fallback else "")
                )

            except Exception as e:
                api_failures += 1
                fallbacks += 1
                active_mode = "nominal"
                print(
                    f"step {step}: Jev error -> nominal mode: {e}"
                )

        # ---------------------------------------------------------
        # 2) CRITICAL FIX:
        #    calculate a FRESH motor action EVERY physics frame.
        # ---------------------------------------------------------
        reference_action = reference.step(obs)

        current_action, effective_mode = apply_jev_mode(
            reference_action,
            obs,
            previous_action,
            active_mode,
        )

        # ---------------------------------------------------------
        # 3) Advance exactly ONE physics frame.
        # ---------------------------------------------------------
        obs, reward, terminated, truncated, info = env.step(
            current_action
        )

        previous_action = current_action.copy()
        total_reward += float(reward)
        recent_rewards.append(float(reward))

        reward_history.append(total_reward)
        speed_history.append(float(obs[2]))
        angle_history.append(float(obs[0]))
        mode_history.append(effective_mode)

        if step % VIDEO_EVERY == 0:
            writer.append_data(env.render())

        if terminated or truncated:
            print(
                f"Episode ended at step {step + 1} | "
                f"terminated={terminated} "
                f"truncated={truncated}"
            )
            break

finally:
    writer.close()
    env.close()

print("\n=== RESULT ===")
print("steps:", len(reward_history))
print("total reward:", round(total_reward, 2))
print("Jev decisions:", len(decision_log))
print("fallbacks:", fallbacks)
print("API failures:", api_failures)

lat = [
    d["latency_ms"]
    for d in decision_log
    if np.isfinite(d.get("latency_ms", np.nan))
]

if lat:
    print(
        "mean Jev latency (ms):",
        round(float(np.mean(lat)), 1)
    )
    print(
        "p95 Jev latency (ms):",
        round(float(np.percentile(lat, 95)), 1)
    )

display(
    Video(
        VIDEO_PATH,
        embed=True,
        html_attributes="controls loop"
    )
)


In [ ]:
#@title 7. Plot diagnostics and save Jev decision log
plt.figure(figsize=(12, 4))
plt.plot(reward_history)
plt.xlabel("Environment step")
plt.ylabel("Cumulative reward")
plt.title("Jev-supervised BipedalWalker — cumulative reward")
plt.grid(True, alpha=0.25)
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(speed_history, label="horizontal speed")
plt.plot(angle_history, label="hull angle")
plt.xlabel("Environment step")
plt.title("Walker dynamics")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()

if decision_log:
    plt.figure(figsize=(12, 4))
    plt.plot(
        [d["step"] for d in decision_log],
        [d.get("confidence", 0.0) for d in decision_log],
        marker="o",
        markersize=3,
    )
    plt.axhline(
        MIN_CONFIDENCE,
        linestyle="--",
        linewidth=1,
        label="fallback threshold",
    )
    plt.ylim(-0.02, 1.02)
    plt.xlabel("Environment step")
    plt.ylabel("Jev confidence")
    plt.title("Jev supervisory confidence")
    plt.legend()
    plt.grid(True, alpha=0.25)
    plt.show()

with open(LOG_PATH, "w", encoding="utf-8") as f:
    json.dump(decision_log, f, indent=2)

print("saved:", LOG_PATH)
print("video:", VIDEO_PATH)


## Why this fixes the jump

The old loop effectively did:

```text
frame 0: choose [hip, knee, hip, knee]
frames 1–9: repeat exactly the same values
```

The fixed loop does:

```text
frame 0:  fresh motor action
frame 1:  fresh motor action
frame 2:  fresh motor action
...
frame 9:  fresh motor action
frame 10: Jev may change supervisory mode
frame 10: fresh motor action under the new mode
```

So `DECISION_EVERY=10` no longer means "hold the motors for 10 frames."

For tighter Jev intervention, try `DECISION_EVERY = 5`.
